# Modelos C y D — Mask R-CNN ResNet50-FPN (Detectron2)
## Dataset Filtrado vs Dataset Completo

| | Modelo A | Modelo B | Modelo B+ | **Modelo C** | **Modelo D** |
|---|---|---|---|---|---|
| **Arquitectura** | YOLOv8m | YOLOv8m | YOLOv8m+PP | **MaskRCNN** | **MaskRCNN** |
| **Dataset** | Completo 174 | Filtrado 101 | Filtrado 101 | **Filtrado 101** | **Completo 174** |
| **Dice previo** | 0.29 | 0.39 | 0.37 | **?** | **?** |

**Hipótesis que contrasta C vs D:**
- Modelo C: MaskRCNN con dataset filtrado — ¿la calidad de las anotaciones supera la cantidad?
- Modelo D: MaskRCNN con dataset completo — ¿MaskRCNN maneja mejor las columnas parciales que YOLOv8?

**Fuente:** máscaras PNG multiclase IDs 0-17 — mismo origen que notebooks anteriores.

## 0 — Instalación

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
print(f'GPU:     {torch.cuda.get_device_name(0)}')
print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
import detectron2
print(f'Detectron2: {detectron2.__version__}')

In [ ]:
import os, json, shutil, random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
import detectron2.data as d2_data
import detectron2.data.transforms as T

from google.colab import drive
drive.mount('/content/drive')

# ── RUTAS ─────────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive')
DATASET_ROOT = DRIVE_ROOT / 'Scoliosis_Dataset'
CSV_PATH     = DATASET_ROOT / 'indice_dataset.csv'
YOLO_DS_FILT = Path('/content/yolo_spine/dataset_filtered')
WORK_DIR     = Path('/content/maskrcnn_spine')
COCO_DIR     = WORK_DIR / 'coco_splits'
OUTPUT_C     = WORK_DIR / 'output_C_filtrado'   # MaskRCNN dataset filtrado
OUTPUT_D     = WORK_DIR / 'output_D_completo'   # MaskRCNN dataset completo

# ── COLUMNAS CSV ──────────────────────────────────────────────
COL_SPLIT = 'split'
COL_IMAGE = 'radiograph_path'
COL_MASK  = 'multiclass_id_png'

# ── CLASES ────────────────────────────────────────────────────
CLASS_NAMES = [
    'T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12',
    'L1','L2','L3','L4','L5'
]
NUM_CLASSES    = 17
# ID máscara PNG 1-based → category_id COCO 1-based (identidad)
MASK_ID_TO_CAT = {i: i for i in range(1, 18)}
SEED           = 42

random.seed(SEED)
np.random.seed(SEED)
COCO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_C.mkdir(parents=True, exist_ok=True)
OUTPUT_D.mkdir(parents=True, exist_ok=True)
print('✔ Configuración lista')

---
## 1 — Preparar Dataset

In [ ]:
# Split idéntico al notebook anterior (mismo SEED)
df = pd.read_csv(CSV_PATH, sep=';')
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df[COL_SPLIT], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[COL_SPLIT], random_state=SEED
)
for d in [train_df, val_df, test_df]:
    d.reset_index(drop=True, inplace=True)

# ── Dataset C: filtrado (reutiliza stems del Dataset B) ───────
def stems_in_split(yolo_filt, split):
    d = Path(yolo_filt) / 'labels' / split
    return {p.stem for p in d.glob('*.txt')} if d.exists() else set()

def filter_df(df, stems):
    return df[df[COL_IMAGE].apply(
        lambda p: Path(p).stem in stems
    )].reset_index(drop=True)

train_f = filter_df(train_df, stems_in_split(YOLO_DS_FILT, 'train'))
val_f   = filter_df(val_df,   stems_in_split(YOLO_DS_FILT, 'val'))
test_f  = filter_df(test_df,  stems_in_split(YOLO_DS_FILT, 'test'))

# ── Dataset D: completo (todas las imágenes) ──────────────────
train_full = train_df.copy()
val_full   = val_df.copy()
test_full  = test_df.copy()

print(f'Dataset C (filtrado): Train={len(train_f)} Val={len(val_f)} Test={len(test_f)}')
print(f'Dataset D (completo): Train={len(train_full)} Val={len(val_full)} Test={len(test_full)}')

In [ ]:
# ── Construir COCO JSON desde máscaras PNG ────────────────────

CATEGORIES = [{'id': i, 'name': CLASS_NAMES[i-1]} for i in range(1, NUM_CLASSES+1)]

def mask_to_anns(mask_path, image_id, ann_id, min_area=80):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if mask is None: return [], ann_id
    if mask.ndim == 3: mask = mask[:,:,0]
    h, w = mask.shape
    anns = []
    for mid, cid in MASK_ID_TO_CAT.items():
        binary = (mask == mid).astype(np.uint8) * 255
        if binary.sum() == 0: continue
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = [c for c in cnts if cv2.contourArea(c) >= min_area]
        if not cnts: continue
        cnt  = max(cnts, key=cv2.contourArea)
        area = float(cv2.contourArea(cnt))
        x, y, bw, bh = cv2.boundingRect(cnt)
        seg  = [cnt.reshape(-1,2).astype(float).flatten().tolist()]
        anns.append({'id': ann_id, 'image_id': image_id, 'category_id': cid,
                     'segmentation': seg, 'bbox': [float(x),float(y),float(bw),float(bh)],
                     'area': area, 'iscrowd': 0})
        ann_id += 1
    return anns, ann_id


def build_coco(split_df, split_name, ann_id=1):
    images, annotations = [], []
    for img_id, (_, row) in enumerate(split_df.iterrows(), 1):
        ip = DATASET_ROOT / row[COL_IMAGE]
        mp = DATASET_ROOT / row[COL_MASK]
        img = cv2.imread(str(ip))
        if img is None: continue
        h, w = img.shape[:2]
        fname = str(ip.relative_to(DATASET_ROOT))
        images.append({'id': img_id, 'file_name': fname, 'width': w, 'height': h})
        anns, ann_id = mask_to_anns(mp, img_id, ann_id)
        annotations.extend(anns)
    out = COCO_DIR / f'{split_name}.json'
    with open(out, 'w') as f:
        json.dump({'images': images, 'annotations': annotations,
                   'categories': CATEGORIES}, f)
    print(f'  {split_name:6s}: {len(images)} imgs, {len(annotations)} anns')
    return out, ann_id


print('Construyendo COCO JSONs...')
aid = 1
TRAIN_JSON_C, aid = build_coco(train_f,    'train_c', aid)
VAL_JSON_C,   aid = build_coco(val_f,      'val_c',   aid)
TEST_JSON_C,  aid = build_coco(test_f,     'test_c',  aid)
TRAIN_JSON_D, aid = build_coco(train_full, 'train_d', aid)
VAL_JSON_D,   aid = build_coco(val_full,   'val_d',   aid)
TEST_JSON_D,  aid = build_coco(test_full,  'test_d',  aid)
print('✔ Listo')

In [ ]:
# ── Registrar en Detectron2 ───────────────────────────────────
IMAGE_ROOT = str(DATASET_ROOT)
for name, jp in [
    ('spine_train_c', TRAIN_JSON_C), ('spine_val_c', VAL_JSON_C), ('spine_test_c', TEST_JSON_C),
    ('spine_train_d', TRAIN_JSON_D), ('spine_val_d', VAL_JSON_D), ('spine_test_d', TEST_JSON_D),
]:
    if name in DatasetCatalog:
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)
    register_coco_instances(name, {}, str(jp), IMAGE_ROOT)
    MetadataCatalog.get(name).thing_classes = CLASS_NAMES
    print(f'✔ {name}')

# Verificación Dataset C
dicts_c = DatasetCatalog.get('spine_train_c')
dicts_d = DatasetCatalog.get('spine_train_d')
print(f'\nC filtrado  train: {len(dicts_c)} imágenes')
print(f'D completo  train: {len(dicts_d)} imágenes')

In [ ]:
# ── Verificación visual ANTES de entrenar ─────────────────────
# Si las máscaras se ven bien sobre las vértebras → continúa
# Si no → revisa IMAGE_ROOT y los file_name en el JSON
from detectron2.utils.visualizer import Visualizer
meta   = MetadataCatalog.get('spine_train_c')
sample = dicts_c[random.randint(0, len(dicts_c)-1)]
img    = cv2.cvtColor(cv2.imread(sample['file_name']), cv2.COLOR_BGR2RGB)
vis    = Visualizer(img, metadata=meta, scale=0.5).draw_dataset_dict(sample)
plt.figure(figsize=(7, 12))
plt.imshow(vis.get_image())
plt.title(f'{Path(sample["file_name"]).name} — {len(sample["annotations"])} vértebras (Dataset C filtrado)')
plt.axis('off'); plt.tight_layout(); plt.show()

---
## 2 — Entrenamiento

In [ ]:
def build_cfg(output_dir, num_classes, max_iter=6000):
    cfg = get_cfg()
    cfg.merge_from_file(
        model_zoo.get_config_file(
            'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
        )
    )
    cfg.DATASETS.TRAIN = ('spine_train',)
    cfg.DATASETS.TEST  = ('spine_val',)
    cfg.MODEL.WEIGHTS  = model_zoo.get_checkpoint_url(
        'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
    )
    cfg.DATALOADER.NUM_WORKERS       = 2
    cfg.SOLVER.IMS_PER_BATCH         = 2
    cfg.SOLVER.BASE_LR               = 0.002
    cfg.SOLVER.MAX_ITER              = max_iter
    cfg.SOLVER.WARMUP_ITERS          = 500
    cfg.SOLVER.WARMUP_FACTOR         = 1.0 / 500
    cfg.SOLVER.STEPS                 = (int(max_iter*.70), int(max_iter*.88))
    cfg.SOLVER.GAMMA                 = 0.1
    cfg.SOLVER.CHECKPOINT_PERIOD     = 1000
    cfg.SOLVER.WEIGHT_DECAY          = 0.0001
    cfg.SOLVER.MOMENTUM              = 0.9
    cfg.SOLVER.AMP.ENABLED           = True
    cfg.TEST.EVAL_PERIOD             = 1000
    cfg.MODEL.ROI_HEADS.NUM_CLASSES          = num_classes
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST    = 0.25
    cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST      = 0.50
    cfg.MODEL.ANCHOR_GENERATOR.SIZES         = [[16],[32],[64],[128],[256]]
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5,1.0,2.0]]
    cfg.MODEL.BACKBONE.FREEZE_AT             = 0
    cfg.INPUT.MIN_SIZE_TRAIN = (800, 1024)
    cfg.INPUT.MAX_SIZE_TRAIN = 1333
    cfg.INPUT.MIN_SIZE_TEST  = 1024
    cfg.INPUT.MAX_SIZE_TEST  = 1333
    cfg.INPUT.RANDOM_FLIP    = 'horizontal'
    cfg.OUTPUT_DIR           = str(output_dir)
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    return cfg


class SpineTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, 'inference')
        return COCOEvaluator(dataset_name, output_dir=output_folder)

    @classmethod
    def build_train_loader(cls, cfg):
        augs = [
            T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
            T.RandomRotation(angle=[-15, 15]),
            T.RandomBrightness(intensity_min=0.75, intensity_max=1.25),
            T.RandomContrast(intensity_min=0.75,   intensity_max=1.25),
            T.ResizeShortestEdge((800, 1024), max_size=1333, sample_style='choice'),
        ]
        return d2_data.build_detection_train_loader(
            cfg, mapper=d2_data.DatasetMapper(cfg, is_train=True, augmentations=augs)
        )


cfg = build_cfg(OUTPUT_DIR, NUM_CLASSES, max_iter=6000)
print('✔ Config lista')
print(f'  Iteraciones: {cfg.SOLVER.MAX_ITER}')
print(f'  Épocas aprox: {cfg.SOLVER.MAX_ITER//(len(train_f)//cfg.SOLVER.IMS_PER_BATCH)}')

In [ ]:
print('=' * 60)
print('MODELO C — MaskRCNN dataset filtrado (101 imgs)')
print('=' * 60)
cfg_c = build_cfg(OUTPUT_C, NUM_CLASSES, max_iter=6000)
cfg_c.DATASETS.TRAIN = ('spine_train_c',)
cfg_c.DATASETS.TEST  = ('spine_val_c',)
trainer_c = SpineTrainer(cfg_c)
trainer_c.resume_or_load(resume=False)
trainer_c.train()
print(f'✔ Modelo C completado → {OUTPUT_C}')

print('\n' + '=' * 60)
print('MODELO D — MaskRCNN dataset completo (174 imgs)')
print('=' * 60)
cfg_d = build_cfg(OUTPUT_D, NUM_CLASSES, max_iter=6000)
cfg_d.DATASETS.TRAIN = ('spine_train_d',)
cfg_d.DATASETS.TEST  = ('spine_val_d',)
trainer_d = SpineTrainer(cfg_d)
trainer_d.resume_or_load(resume=False)
trainer_d.train()
print(f'✔ Modelo D completado → {OUTPUT_D}')

---
## 3 — Evaluación

In [ ]:
# Cargar ambos modelos
cfg_c_eval = cfg_c.clone()
cfg_c_eval.MODEL.WEIGHTS               = str(OUTPUT_C / 'model_final.pth')
cfg_c_eval.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.25
cfg_c_eval.MODEL.ROI_HEADS.NMS_THRESH_TEST   = 0.50
predictor_c = DefaultPredictor(cfg_c_eval)

cfg_d_eval = cfg_d.clone()
cfg_d_eval.MODEL.WEIGHTS               = str(OUTPUT_D / 'model_final.pth')
cfg_d_eval.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.25
cfg_d_eval.MODEL.ROI_HEADS.NMS_THRESH_TEST   = 0.50
predictor_d = DefaultPredictor(cfg_d_eval)
print('✔ Modelos C y D cargados')

# AP COCO nativo
print('\n=== AP COCO — Modelo C (filtrado) ===')
eval_c = COCOEvaluator('spine_test_c', output_dir=str(OUTPUT_C/'eval'))
print(inference_on_dataset(predictor_c.model,
      build_detection_test_loader(cfg_c_eval,'spine_test_c'), eval_c))

print('\n=== AP COCO — Modelo D (completo) ===')
eval_d = COCOEvaluator('spine_test_d', output_dir=str(OUTPUT_D/'eval'))
print(inference_on_dataset(predictor_d.model,
      build_detection_test_loader(cfg_d_eval,'spine_test_d'), eval_d))

In [ ]:
# ── Dice e IoU por clase ──────────────────────────────────────
def eval_maskrcnn(predictor, split_df, model_name):
    """Evalúa Dice e IoU por clase para un predictor dado."""
    dice_cls = {c: [] for c in range(NUM_CLASSES)}
    iou_cls  = {c: [] for c in range(NUM_CLASSES)}
    l5_log   = []
    for _, row in split_df.iterrows():
        ip  = DATASET_ROOT / row[COL_IMAGE]
        mp  = DATASET_ROOT / row[COL_MASK]
        bgr = cv2.imread(str(ip))
        if bgr is None: continue
        H, W = bgr.shape[:2]
        gt   = gt_from_png(mp, H, W)
        out  = predictor(bgr)
        inst = out['instances'].to('cpu')
        pred = {c: np.zeros((H,W),np.uint8) for c in range(NUM_CLASSES)}
        if len(inst) > 0:
            ca, ma, sa = inst.pred_classes.numpy(), inst.pred_masks.numpy(), inst.scores.numpy()
            for c in range(NUM_CLASSES):
                idx = np.where(ca==c)[0]
                if len(idx)==0: continue
                pred[c] = ma[idx[np.argmax(sa[idx])]].astype(np.uint8)
        stem = ip.stem
        for c in range(NUM_CLASSES):
            if gt[c].sum()==0: continue
            d, iou = dice_iou(pred[c], gt[c])
            dice_cls[c].append(d); iou_cls[c].append(iou)
            if c==16:
                l5_log.append({'image':ip.name,'model':model_name,
                               'split':'Scoliosis' if stem.startswith('S_') else 'Normal',
                               'dice':d,'iou':iou,'gt_px':int(gt[c].sum()),
                               'pred_px':int(pred[c].sum()),'detected':pred[c].sum()>0})
    return dice_cls, iou_cls, l5_log


# Evaluar C sobre test filtrado y D sobre test completo
# IMPORTANTE: ambos se evalúan sobre el mismo test_f (filtrado)
# para comparación justa — D también se evalúa en imágenes completas
print('Evaluando Modelo C (MaskRCNN filtrado)...')
dice_c, iou_c, l5_log_c = eval_maskrcnn(predictor_c, test_f, 'C_MaskRCNN_filt')

print('Evaluando Modelo D (MaskRCNN completo)...')
dice_d, iou_d, l5_log_d = eval_maskrcnn(predictor_d, test_f, 'D_MaskRCNN_full')

print('✔ Evaluación completada')

In [ ]:
# ── Tabla comparativa A vs B vs B+ vs C ──────────────────────
prev_csv = DRIVE_ROOT / 'models' / 'comparacion_3modelos.csv'
if prev_csv.exists():
    prev      = pd.read_csv(prev_csv)
    dice_a_v  = dict(zip(range(NUM_CLASSES), prev['dice_A'].values))
    dice_b_v  = dict(zip(range(NUM_CLASSES), prev['dice_B'].values))
    dice_bp_v = dict(zip(range(NUM_CLASSES), prev['dice_Bplus'].values))
    print('✔ A/B/B+ cargados desde CSV')
else:
    dice_a_v  = dict(zip(range(17),[0.7044,0.6511,0.4522,0.3042,0.3174,0.2620,
                                     0.2250,0.1543,0.1328,0.1670,0.0589,0.0925,
                                     0.1416,0.2121,0.3132,0.5111,0.2434]))
    dice_b_v  = dict(zip(range(17),[0.7175,0.5860,0.4112,0.3048,0.3467,0.3218,
                                     0.3279,0.2988,0.3254,0.2820,0.3091,0.3380,
                                     0.3580,0.3989,0.4532,0.5111,0.3031]))
    dice_bp_v = dict(zip(range(17),[0.6691,0.5908,0.4279,0.2903,0.2888,0.3075,
                                     0.3033,0.2974,0.2997,0.2979,0.3026,0.3148,
                                     0.3819,0.3502,0.4131,0.5111,0.3126]))
    print('⚠ Usando valores de respaldo')

print(f'\n{"Clase":<6} {"A":>7} {"B":>7} {"B+":>7} {"C-Filt":>8} {"D-Full":>8} {"Mejor":>6}')
print('─' * 56)
rows_5 = []
for c in range(NUM_CLASSES):
    da  = dice_a_v.get(c, 0.0)
    db  = dice_b_v.get(c, 0.0)
    dbp = dice_bp_v.get(c, 0.0)
    dc  = np.mean(dice_c[c]) if dice_c[c] else 0.0
    dd  = np.mean(dice_d[c]) if dice_d[c] else 0.0
    bl  = ['A','B','B+','C','D'][[da,db,dbp,dc,dd].index(max(da,db,dbp,dc,dd))]
    tag = ' ◄L5' if c==16 else ''
    print(f'{CLASS_NAMES[c]:<6} {da:>7.4f} {db:>7.4f} {dbp:>7.4f} {dc:>8.4f} {dd:>8.4f} {bl:>6}{tag}')
    rows_5.append({'clase':CLASS_NAMES[c],'dice_A':da,'dice_B':db,'dice_Bplus':dbp,
                   'dice_C':dc,'dice_D':dd,'mejor':bl,
                   'iou_C':np.mean(iou_c[c]) if iou_c[c] else 0.0,
                   'iou_D':np.mean(iou_d[c]) if iou_d[c] else 0.0})
print('─' * 56)
ma  = np.mean([r['dice_A']     for r in rows_5])
mb  = np.mean([r['dice_B']     for r in rows_5])
mbp = np.mean([r['dice_Bplus'] for r in rows_5])
mc  = np.mean([r['dice_C']     for r in rows_5])
md  = np.mean([r['dice_D']     for r in rows_5])
print(f'{"MEAN":<6} {ma:>7.4f} {mb:>7.4f} {mbp:>7.4f} {mc:>8.4f} {md:>8.4f}')
print(f'\n  A  YOLOv8  completo     : {ma:.4f}')
print(f'  B  YOLOv8  filtrado     : {mb:.4f}')
print(f'  B+ YOLOv8  filt+pos     : {mbp:.4f}')
print(f'  C  MaskRCNN filtrado    : {mc:.4f}  Δ vs B: {mc-mb:+.4f}')
print(f'  D  MaskRCNN completo    : {md:.4f}  Δ vs A: {md-ma:+.4f}')
print(f'  Paper anterior          : 0.7400')
print(f'\n  C vs D (filtrado vs completo en MaskRCNN): {mc-md:+.4f}')
pd.DataFrame(rows_5).to_csv(DRIVE_ROOT/'models'/'comparacion_5modelos.csv', index=False)
print('\n✔ Tabla guardada')

In [ ]:
# ── Gráfico comparativo ───────────────────────────────────────
fig = plt.figure(figsize=(24, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
x = np.arange(NUM_CLASSES); w = 0.16

# Dice por clase — 5 modelos
ax1 = fig.add_subplot(gs[0,:])
for off, (lbl, col, vals) in enumerate([
    ('A YOLOv8 completo',    '#3498db', [r['dice_A']     for r in rows_5]),
    ('B YOLOv8 filtrado',    '#2ecc71', [r['dice_B']     for r in rows_5]),
    ('B+ YOLOv8 filt+pos',   '#e67e22', [r['dice_Bplus'] for r in rows_5]),
    ('C MaskRCNN filtrado',  '#9b59b6', [r['dice_C']     for r in rows_5]),
    ('D MaskRCNN completo',  '#e74c3c', [r['dice_D']     for r in rows_5]),
]):
    ax1.bar(x+(off-2)*w, vals, w, label=lbl, color=col, alpha=0.85, edgecolor='k')
ax1.axhline(0.70, color='red',  ls='--', lw=1.5, label='Umbral 0.70')
ax1.axhline(0.74, color='gray', ls=':',  lw=1.5, label='Paper 0.74')
ax1.set_xticks(x); ax1.set_xticklabels(CLASS_NAMES, rotation=45)
ax1.set_ylabel('Dice'); ax1.set_ylim(0, 1.1)
ax1.set_title('Dice por vértebra — Cinco estrategias', fontsize=13)
ax1.legend(fontsize=8, ncol=3); ax1.grid(axis='y', alpha=0.3)

# Resumen por región
ax2 = fig.add_subplot(gs[1,0])
groups = {'T1-T6':range(0,6),'T7-T12':range(6,12),'L1-L5':range(12,17),'GLOBAL':range(0,17)}
xg = np.arange(len(groups)); wg = 0.15
dc_v = {c: np.mean(dice_c[c]) if dice_c[c] else 0.0 for c in range(NUM_CLASSES)}
dd_v = {c: np.mean(dice_d[c]) if dice_d[c] else 0.0 for c in range(NUM_CLASSES)}
for off2, (lbl, col, vd) in enumerate([
    ('A','#3498db',dice_a_v), ('B','#2ecc71',dice_b_v),
    ('B+','#e67e22',dice_bp_v), ('C','#9b59b6',dc_v), ('D','#e74c3c',dd_v)
]):
    yv = [np.mean([vd.get(c,0) for c in rng]) for rng in groups.values()]
    bars = ax2.bar(xg+(off2-2)*wg, yv, wg, label=lbl, color=col, alpha=0.85, edgecolor='k')
    for bar, v in zip(bars, yv):
        ax2.text(bar.get_x()+bar.get_width()/2, v+0.01,
                 f'{v:.2f}', ha='center', fontsize=6, rotation=90)
ax2.axhline(0.70,color='red',ls='--',lw=1.5); ax2.axhline(0.74,color='gray',ls=':',lw=1.5)
ax2.set_xticks(xg); ax2.set_xticklabels(list(groups.keys()))
ax2.set_ylabel('Dice promedio'); ax2.set_ylim(0,1.1)
ax2.set_title('Dice por región anatómica', fontsize=11)
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# Delta C vs D (filtrado vs completo en MaskRCNN)
ax3 = fig.add_subplot(gs[1,1])
delta_cd = [rows_5[c]['dice_C']-rows_5[c]['dice_D'] for c in range(NUM_CLASSES)]
ax3.bar(x, delta_cd, color=['#9b59b6' if d>=0 else '#e74c3c' for d in delta_cd],
        edgecolor='k', alpha=0.85)
ax3.axhline(0, color='black', lw=1.5)
ax3.set_xticks(x); ax3.set_xticklabels(CLASS_NAMES, rotation=45)
ax3.set_ylabel('Δ Dice (C_filt − D_full)')
ax3.set_title('MaskRCNN: Filtrado vs Completo\nMorado=Filtrado mejor, Rojo=Completo mejor',
              fontsize=11)
ax3.grid(axis='y', alpha=0.3)

plt.suptitle('Comparación Final — A|B|B+ (YOLOv8) vs C|D (MaskRCNN)',
             fontsize=12, fontweight='bold')
plt.savefig(str(DRIVE_ROOT/'models'/'comparacion_5modelos.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✔ Gráfico guardado')

In [ ]:
# ── Análisis L5 ───────────────────────────────────────────────
if l5_log_c or l5_log_d:
    l5_cd = pd.DataFrame(l5_log_c + l5_log_d)
    print('═══ L5 — Modelo C vs Modelo D ════════════════════════')
    print(l5_cd.groupby('model')[['dice','detected']].agg(
        {'dice':['mean','median','std'],'detected':'mean'}
    ).round(4).to_string())
    print()
    print('Por tipo de columna:')
    print(l5_cd.groupby(['model','split'])['dice'].agg(
        ['mean','median','std','count']
    ).round(4).to_string())

    prev_l5 = DRIVE_ROOT / 'models' / 'l5_comparacion.csv'
    l5_all  = pd.concat([pd.read_csv(prev_l5), l5_cd], ignore_index=True) \
              if prev_l5.exists() else l5_cd

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = {'A_completo':'#3498db','B_filtrado':'#2ecc71','B+':'#e67e22',
              'C_MaskRCNN_filt':'#9b59b6','D_MaskRCNN_full':'#e74c3c'}
    for mdl in l5_all['model'].unique():
        sub = l5_all[l5_all['model']==mdl]['dice']
        axes[0].hist(sub, bins=10, alpha=0.5, edgecolor='k',
                     color=colors.get(mdl,'gray'),
                     label=f'{mdl} μ={sub.mean():.3f}')
    axes[0].axvline(0.52,color='gray',ls=':',lw=1.5,label='Paper=0.52')
    axes[0].axvline(0.70,color='red', ls='--',lw=1.5,label='Umbral=0.70')
    axes[0].set_xlabel('Dice L5'); axes[0].set_title('L5 todos los modelos')
    axes[0].legend(fontsize=7)

    # Boxplot C vs D por tipo
    for ax_idx, (mdl, col) in enumerate([
        ('C_MaskRCNN_filt','#9b59b6'), ('D_MaskRCNN_full','#e74c3c')
    ]):
        sub_m = l5_cd[l5_cd['model']==mdl]
        bp = axes[ax_idx+1].boxplot(
            [sub_m[sub_m['split']=='Normal']['dice'].values,
             sub_m[sub_m['split']=='Scoliosis']['dice'].values],
            labels=['Normal','Scoliosis'], patch_artist=True
        )
        for box in bp['boxes']:
            box.set_facecolor(col); box.set_alpha(0.7)
        axes[ax_idx+1].axhline(0.70,color='red',ls='--',lw=1.5)
        axes[ax_idx+1].set_title(f'{mdl}\nL5 Normal vs Scoliosis')
        axes[ax_idx+1].set_ylabel('Dice L5')

    plt.suptitle('Análisis L5 — C (filtrado) vs D (completo)', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(DRIVE_ROOT/'models'/'l5_5modelos.png'), dpi=150, bbox_inches='tight')
    plt.show()
    l5_all.to_csv(DRIVE_ROOT/'models'/'l5_5modelos.csv', index=False)
    print('✔ Análisis L5 guardado')

In [ ]:
# ── Visualización cualitativa ─────────────────────────────────
def show_pred(predictor, ip, mp, title=''):
    bgr = cv2.imread(str(ip))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    H,W = rgb.shape[:2]
    rng = np.random.RandomState(0)
    pal = rng.randint(60, 230, (NUM_CLASSES,3), dtype=np.uint8)
    gt  = gt_from_png(mp, H, W)

    def ov(md):
        o = rgb.copy()
        for c, m in md.items():
            if m.sum()==0: continue
            col = tuple(int(x) for x in pal[c])
            cl  = np.zeros_like(rgb); cl[m==1]=col
            o   = cv2.addWeighted(o,.65,cl,.35,0)
            cnts,_ = cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(o,cnts,-1,col,2)
            M = cv2.moments(m)
            if M['m00']>0:
                cx,cy=int(M['m10']/M['m00']),int(M['m01']/M['m00'])
                cv2.putText(o,CLASS_NAMES[c],(cx-12,cy+5),
                            cv2.FONT_HERSHEY_SIMPLEX,.45,(255,255,255),1)
        return o

    out  = predictor(bgr)
    inst = out['instances'].to('cpu')
    pred = {c:np.zeros((H,W),np.uint8) for c in range(NUM_CLASSES)}
    if len(inst)>0:
        ca,ma,sa = inst.pred_classes.numpy(),inst.pred_masks.numpy(),inst.scores.numpy()
        for c in range(NUM_CLASSES):
            idx=np.where(ca==c)[0]
            if len(idx)==0: continue
            pred[c]=ma[idx[np.argmax(sa[idx])]].astype(np.uint8)

    fig,axes=plt.subplots(1,3,figsize=(18,9))
    axes[0].imshow(rgb);     axes[0].set_title('Original')
    axes[1].imshow(ov(gt));  axes[1].set_title('Ground Truth')
    axes[2].imshow(ov(pred));axes[2].set_title('Mask R-CNN R50-FPN')
    for ax in axes: ax.axis('off')
    plt.suptitle(title or ip.name, fontsize=11)
    plt.tight_layout(); plt.show()


# 4 ejemplos: 2 Normal, 2 Scoliosis — comparando C vs D
n_rows = test_f[test_f[COL_IMAGE].str.startswith('Normal')]
s_rows = test_f[test_f[COL_IMAGE].str.startswith('Scoliosis')]
samples = pd.concat([
    n_rows.sample(min(2,len(n_rows)),   random_state=SEED),
    s_rows.sample(min(2,len(s_rows)),   random_state=SEED)
])

for _,row in samples.iterrows():
    ip  = DATASET_ROOT/row[COL_IMAGE]
    mp  = DATASET_ROOT/row[COL_MASK]
    bgr = cv2.imread(str(ip))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    H, W = rgb.shape[:2]
    rng  = np.random.RandomState(0)
    pal  = rng.randint(60, 230, (NUM_CLASSES,3), dtype=np.uint8)
    gt   = gt_from_png(mp, H, W)

    def ov(md):
        o = rgb.copy()
        for c, m in md.items():
            if m.sum()==0: continue
            col = tuple(int(x) for x in pal[c])
            cl  = np.zeros_like(rgb); cl[m==1]=col
            o   = cv2.addWeighted(o,.65,cl,.35,0)
            cnts,_ = cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(o,cnts,-1,col,2)
            M = cv2.moments(m)
            if M['m00']>0:
                cx,cy=int(M['m10']/M['m00']),int(M['m01']/M['m00'])
                cv2.putText(o,CLASS_NAMES[c],(cx-12,cy+5),
                            cv2.FONT_HERSHEY_SIMPLEX,.4,(255,255,255),1)
        return o

    def pred_masks(predictor):
        out  = predictor(bgr)
        inst = out['instances'].to('cpu')
        pred = {c:np.zeros((H,W),np.uint8) for c in range(NUM_CLASSES)}
        if len(inst)>0:
            ca,ma,sa = inst.pred_classes.numpy(),inst.pred_masks.numpy(),inst.scores.numpy()
            for c in range(NUM_CLASSES):
                idx=np.where(ca==c)[0]
                if len(idx)==0: continue
                pred[c]=ma[idx[np.argmax(sa[idx])]].astype(np.uint8)
        return pred

    tipo = 'Normal' if 'Normal' in row[COL_IMAGE] else 'Scoliosis'
    fig, axes = plt.subplots(1, 4, figsize=(24, 9))
    axes[0].imshow(rgb);                  axes[0].set_title('Original')
    axes[1].imshow(ov(gt));               axes[1].set_title('Ground Truth')
    axes[2].imshow(ov(pred_masks(predictor_c))); axes[2].set_title('C — MaskRCNN filtrado')
    axes[3].imshow(ov(pred_masks(predictor_d))); axes[3].set_title('D — MaskRCNN completo')
    for ax in axes: ax.axis('off')
    plt.suptitle(f'{ip.name} ({tipo})', fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Guardar modelo y resumen ──────────────────────────────────
save_dir = DRIVE_ROOT / 'models'
os.makedirs(save_dir, exist_ok=True)
dst_c = save_dir / 'maskrcnn_C_filtrado_final.pth'
dst_d = save_dir / 'maskrcnn_D_completo_final.pth'
shutil.copy(OUTPUT_C / 'model_final.pth', dst_c)
shutil.copy(OUTPUT_D / 'model_final.pth', dst_d)
print(f'✔ C guardado: {dst_c.name}  ({dst_c.stat().st_size/1e6:.0f} MB)')
print(f'✔ D guardado: {dst_d.name}  ({dst_d.stat().st_size/1e6:.0f} MB)')

def rmean(vd, rng):
    return np.mean([vd.get(c,0) for c in rng])

l5_c_dice = pd.DataFrame(l5_log_c)['dice'].mean() if l5_log_c else 0.0
l5_d_dice = pd.DataFrame(l5_log_d)['dice'].mean() if l5_log_d else 0.0

print('\n' + '='*68)
print('RESUMEN FINAL — Cinco modelos')
print('='*68)
print(f"""
A  YOLOv8m completo (174):          Dice={ma:.4f}
B  YOLOv8m filtrado (101):          Dice={mb:.4f}
B+ YOLOv8m filtrado+posicional:     Dice={mbp:.4f}
C  MaskRCNN R50-FPN filtrado (101): Dice={mc:.4f}  Δ vs B: {mc-mb:+.4f}
D  MaskRCNN R50-FPN completo (174): Dice={md:.4f}  Δ vs A: {md-ma:+.4f}
Paper anterior (YOLOv8m, 134 imgs): Dice=0.7400

Impacto del filtrado:
  YOLOv8   (A→B): {mb-ma:+.4f}
  MaskRCNN (D→C): {mc-md:+.4f}
  → ¿Qué arquitectura se beneficia más del filtrado?

Dice L5:
  C MaskRCNN filtrado: {l5_c_dice:.4f}
  D MaskRCNN completo: {l5_d_dice:.4f}
  Paper anterior:      0.5200

Región lumbar (L1-L5):
  C: {rmean(dc_v, range(12,17)):.4f}
  D: {rmean(dd_v, range(12,17)):.4f}
  B: {rmean(dice_b_v, range(12,17)):.4f}
""")
print('='*68)